# GeoDiff-GAN on DGX A100: Three-Model Comparison

Single-GPU workflow for `twilight2002/sentinel2-bharat`: resumable download and preparation, quarantine, deterministic spatial-block 80/10/10 split within each tile, caption-free training, and comparison of small-improved, medium, and large.

**Every variant uses resize-convolution. PixelShuffle is prohibited.**

> Patch splits within the same tile are useful for controlled model comparison but do not prove geographic generalization. Final paper results should also use unseen MGRS tiles.


In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys, json, time

# Change this only if the supervisor assigned a different physical GPU.
ALLOCATED_GPU_INDEX = "0"
os.environ["CUDA_VISIBLE_DEVICES"] = ALLOCATED_GPU_INDEX
os.environ["PYTHONUNBUFFERED"] = "1"

HOME = Path.home()
THESIS_ROOT = HOME / "geodiff_dgx"
REPOSITORY_DIR = THESIS_ROOT / "geodiff-gan"
DOWNLOAD_ROOT = THESIS_ROOT / "downloads"
DATASET_ROOT = THESIS_ROOT / "datasets" / "sentinel2-bharat"
WORK_ROOT = THESIS_ROOT / "geodiff-output"
REPOSITORY_URL = "https://github.com/shashankjs2002/SI-SR-1.git"
KAGGLE_DATASET = "twilight2002/sentinel2-bharat"

for path in (THESIS_ROOT, DOWNLOAD_ROOT, DATASET_ROOT, WORK_ROOT):
    path.mkdir(parents=True, exist_ok=True)

def run(command, cwd=None, env=None):
    command = [str(value) for value in command]
    print("+", " ".join(command), flush=True)
    subprocess.run(command, cwd=cwd, check=True, env=env)

print("THESIS_ROOT:", THESIS_ROOT)
run(["df", "-h", str(THESIS_ROOT)])
run(["nvidia-smi", "-L"])


## 1. Clone/update and install

Existing files are never deleted automatically. The notebook stops if the target repository directory exists but is not a Git clone.


In [ ]:
from pathlib import Path
import os, sys

if REPOSITORY_DIR.exists():
    if not (REPOSITORY_DIR / ".git").exists():
        raise RuntimeError(f"{REPOSITORY_DIR} exists but is not a Git clone; rename it manually.")
    run(["git", "pull", "--ff-only"], cwd=REPOSITORY_DIR)
else:
    run(["git", "clone", "--depth", "1", REPOSITORY_URL, REPOSITORY_DIR])

# Install into the active Jupyter kernel; do not reinstall CUDA PyTorch.
PYTHON = Path(sys.executable)
PIP = [PYTHON, "-m", "pip"]
run([*PIP, "install", "--upgrade", "pip", "setuptools", "wheel"])
run([*PIP, "install", "numpy>=1.26", "Pillow>=10", "PyYAML>=6", "tqdm>=4.66", "rasterio>=1.3", "pandas", "matplotlib", "kaggle"])
run([*PIP, "install", "-e", ".", "--no-deps"], cwd=REPOSITORY_DIR)
sys.path.insert(0, str(REPOSITORY_DIR / "src"))
os.chdir(REPOSITORY_DIR)
run(["git", "log", "-1", "--oneline"], cwd=REPOSITORY_DIR)


In [ ]:
import torch
print("PyTorch:", torch.__version__, "CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available(), "visible GPUs:", torch.cuda.device_count())
if not torch.cuda.is_available():
    raise RuntimeError("CUDA unavailable; ask the administrator to check the Jupyter container.")
if torch.cuda.device_count() != 1:
    raise RuntimeError("Exactly one GPU must be visible. Restart the kernel after setting ALLOCATED_GPU_INDEX.")
props = torch.cuda.get_device_properties(0)
print("GPU:", props.name, "VRAM GiB:", props.total_memory / 1024**3)


## 2. Authenticate and download from your Kaggle account

Uses `~/.kaggle/kaggle.json`, or securely asks for username/API key. `curl -C -` resumes a partial archive and retries network failures.


In [ ]:
from getpass import getpass
import json, os

KAGGLE_DIR = HOME / ".kaggle"
KAGGLE_JSON = KAGGLE_DIR / "kaggle.json"
KAGGLE_NETRC = KAGGLE_DIR / "kaggle.netrc"
KAGGLE_DIR.mkdir(parents=True, exist_ok=True)
if not KAGGLE_JSON.exists():
    username = input("Kaggle username: ").strip()
    key = getpass("Kaggle API key: ").strip()
    if not username or not key: raise RuntimeError("Kaggle credentials not provided")
    KAGGLE_JSON.write_text(json.dumps({"username":username,"key":key}), encoding="utf-8")
    KAGGLE_JSON.chmod(0o600)
credentials = json.loads(KAGGLE_JSON.read_text(encoding="utf-8"))
KAGGLE_NETRC.write_text(f"machine www.kaggle.com login {credentials['username']} password {credentials['key']}\n",encoding="utf-8")
KAGGLE_NETRC.chmod(0o600)
print("Kaggle user:", credentials["username"], "(secret will not be printed)")


In [ ]:
import zipfile

DOWNLOAD_DATASET = True
ARCHIVE = DOWNLOAD_ROOT / "sentinel2-bharat.zip"
EXTRACT_MARKER = DATASET_ROOT / ".kaggle_extract_complete"

def archive_is_complete(path):
    if not path.exists(): return False
    try:
        with zipfile.ZipFile(path) as candidate:
            return candidate.testzip() is None
    except zipfile.BadZipFile:
        return False

if DOWNLOAD_DATASET and not EXTRACT_MARKER.exists():
    if archive_is_complete(ARCHIVE):
        print("Reusing complete downloaded archive:", ARCHIVE)
    else:
        url = "https://www.kaggle.com/api/v1/datasets/download/twilight2002/sentinel2-bharat"
        # Resume a partial archive and retry network failures without exposing the API key.
        run(["curl","-L","--fail","--retry","20","--retry-delay","10","--retry-all-errors","-C","-","--netrc-file",KAGGLE_NETRC,"-o",ARCHIVE,url])
    if not archive_is_complete(ARCHIVE): raise RuntimeError(f"Downloaded archive is incomplete: {ARCHIVE}")
    with zipfile.ZipFile(ARCHIVE) as archive: archive.extractall(DATASET_ROOT)
    EXTRACT_MARKER.write_text("complete\n", encoding="utf-8")
    ARCHIVE.unlink(missing_ok=True)
else:
    print("Reusing extracted dataset:", DATASET_ROOT)
print("SAFE directories:", len(list(DATASET_ROOT.rglob("*.SAFE"))))
run(["du","-sh",DATASET_ROOT])


## 3. Incremental Sentinel-2 preparation

Canonical SAFE discovery ignores nested wrapper duplicates. The preparation state skips completed products and processes newly added products on rerun.


In [ ]:
from pathlib import Path
from collections import Counter
import json
from geodiff_gan.data import sentinel as sentinel_module

PATCH_SIZE = 512
PATCH_STRIDE = 384
MINIMUM_VALID_FRACTION = 0.95
MAX_PRODUCTS = None
REBUILD_PATCHES = False
PATCH_ROOT = WORK_ROOT / "patches"
RAW_MANIFEST = WORK_ROOT / "manifest_raw.jsonl"
PREPARATION_STATE = WORK_ROOT / "preparation-state.json"

safe_candidates = sorted(DATASET_ROOT.rglob("*.SAFE"))
safe_products = sentinel_module.discover_safe_products(DATASET_ROOT)
print(f"Canonical={len(safe_products)}, scanned={len(safe_candidates)}, wrappers ignored={len(safe_candidates)-len(safe_products)}")
for product in safe_products[:20]: print(" -", sentinel_module.source_product_name(product), "->", product)
if not safe_products: raise RuntimeError(f"No canonical SAFE product below {DATASET_ROOT}")

command = [PYTHON,"-m","geodiff_gan.cli.prepare_sentinel","--input",DATASET_ROOT,"--output",PATCH_ROOT,"--manifest",RAW_MANIFEST,"--state",PREPARATION_STATE,"--patch-size",PATCH_SIZE,"--stride",PATCH_STRIDE,"--minimum-valid-fraction",MINIMUM_VALID_FRACTION,"--unmatched-split","train"]
if MAX_PRODUCTS is not None: command.extend(["--max-products",MAX_PRODUCTS])
if REBUILD_PATCHES: command.append("--rebuild")
for historical in (WORK_ROOT/"manifest.before-edge-filter.jsonl", WORK_ROOT/"rejected-edge-patches.jsonl"):
    if historical.exists(): command.extend(["--completed-manifest",historical])
run(command, cwd=REPOSITORY_DIR)
records = [json.loads(line) for line in RAW_MANIFEST.read_text(encoding="utf-8").splitlines() if line.strip()]
print("Patches:",len(records),"products:",len({r['source_product'] for r in records}),"tiles:",Counter(r['tile_id'] for r in records))


## 4. Quarantine black/corrupt edge patches

Rejected binaries are retained under `quarantine_edge_patches` with rejection reasons.


In [ ]:
from pathlib import Path
from collections import Counter
import json
import shutil
import numpy as np

WORK_ROOT = globals().get("WORK_ROOT", THESIS_ROOT / "geodiff-output")
MANIFEST = RAW_MANIFEST
FILTER_EDGE_PATCHES = globals().get("FILTER_EDGE_PATCHES", True)
BLACK_THRESHOLD = globals().get("BLACK_THRESHOLD", 1e-6)
MAX_EDGE_BLACK_FRACTION = globals().get("MAX_EDGE_BLACK_FRACTION", 0.002)
MAX_CORNER_BLACK_FRACTION = globals().get("MAX_CORNER_BLACK_FRACTION", 0.002)
MAX_OVERALL_BLACK_FRACTION = globals().get("MAX_OVERALL_BLACK_FRACTION", 0.005)
EDGE_WIDTH = globals().get("EDGE_WIDTH", 32)
CORNER_SIZE = globals().get("CORNER_SIZE", 64)

BACKUP_MANIFEST = WORK_ROOT / "manifest.before-edge-filter.jsonl"
REJECTED_MANIFEST = WORK_ROOT / "rejected-edge-patches.jsonl"
QUARANTINE_ROOT = WORK_ROOT / "quarantine_edge_patches"

if not MANIFEST.exists():
    raise FileNotFoundError(f"Manifest not found: {MANIFEST}")
active_records = [
    json.loads(line)
    for line in MANIFEST.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

backup_records = []
if BACKUP_MANIFEST.exists():
    backup_records = [
        json.loads(line)
        for line in BACKUP_MANIFEST.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
backup_by_patch = {
    record.get("original_patch", record["patch"]): record
    for record in backup_records
}
backup_count_before = len(backup_by_patch)
for record in active_records:
    backup_by_patch.setdefault(record["patch"], record)
backup_records = list(backup_by_patch.values())
BACKUP_MANIFEST.write_text(
    "".join(json.dumps(record) + "\n" for record in backup_records),
    encoding="utf-8",
)
print(
    "Updated pre-filter audit manifest:", BACKUP_MANIFEST,
    f"(+{len(backup_by_patch) - backup_count_before} records)",
)

existing_rejected = []
if REJECTED_MANIFEST.exists():
    existing_rejected = [
        json.loads(line)
        for line in REJECTED_MANIFEST.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
rejected_keys = {
    record.get("original_patch", record.get("patch"))
    for record in existing_rejected
}

def inspect_black_background(path):
    path = Path(path)
    if not path.exists():
        return True, {
            "reason": "missing_patch",
            "overall_black_fraction": 1.0,
            "edge_black_fraction": 1.0,
            "corner_black_fraction": 1.0,
        }
    try:
        with np.load(path) as data:
            hr = np.asarray(data["hr"], dtype=np.float32)
    except Exception as error:
        return True, {
            "reason": f"unreadable_patch:{type(error).__name__}",
            "overall_black_fraction": 1.0,
            "edge_black_fraction": 1.0,
            "corner_black_fraction": 1.0,
        }
    if hr.ndim != 3 or hr.shape[0] != 3:
        return True, {
            "reason": f"unexpected_shape:{tuple(hr.shape)}",
            "overall_black_fraction": 1.0,
            "edge_black_fraction": 1.0,
            "corner_black_fraction": 1.0,
        }
    if not np.isfinite(hr).all():
        return True, {
            "reason": "nonfinite_pixels",
            "overall_black_fraction": 1.0,
            "edge_black_fraction": 1.0,
            "corner_black_fraction": 1.0,
        }

    black = np.all(hr <= BLACK_THRESHOLD, axis=0)
    height, width = black.shape
    edge = min(EDGE_WIDTH, height // 2, width // 2)
    corner = min(CORNER_SIZE, height // 2, width // 2)
    edge_pixels = np.concatenate([
        black[:edge].ravel(),
        black[-edge:].ravel(),
        black[edge:-edge, :edge].ravel(),
        black[edge:-edge, -edge:].ravel(),
    ])
    corner_pixels = np.concatenate([
        black[:corner, :corner].ravel(),
        black[:corner, -corner:].ravel(),
        black[-corner:, :corner].ravel(),
        black[-corner:, -corner:].ravel(),
    ])
    stats = {
        "overall_black_fraction": float(black.mean()),
        "edge_black_fraction": float(edge_pixels.mean()),
        "corner_black_fraction": float(corner_pixels.mean()),
    }
    reasons = []
    if stats["overall_black_fraction"] > MAX_OVERALL_BLACK_FRACTION:
        reasons.append("overall_black")
    if stats["edge_black_fraction"] > MAX_EDGE_BLACK_FRACTION:
        reasons.append("edge_black")
    if stats["corner_black_fraction"] > MAX_CORNER_BLACK_FRACTION:
        reasons.append("corner_black")
    stats["reason"] = ",".join(reasons) if reasons else "accepted"
    return bool(reasons), stats

kept_records = []
newly_rejected = []
if FILTER_EDGE_PATCHES:
    for index, record in enumerate(active_records, start=1):
        original_patch = record["patch"]
        reject, stats = inspect_black_background(original_patch)
        if reject:
            rejected_record = dict(record)
            rejected_record["original_patch"] = original_patch
            rejected_record["filter"] = stats
            source_path = Path(original_patch)
            if source_path.exists():
                destination = QUARANTINE_ROOT / record["tile_id"] / source_path.name
                destination.parent.mkdir(parents=True, exist_ok=True)
                if source_path.resolve() != destination.resolve():
                    shutil.move(str(source_path), str(destination))
                rejected_record["patch"] = str(destination)
            if original_patch not in rejected_keys:
                newly_rejected.append(rejected_record)
                rejected_keys.add(original_patch)
        else:
            kept_records.append(record)
        if index % 200 == 0 or index == len(active_records):
            print(f"Scanned {index}/{len(active_records)} patches")
else:
    kept_records = active_records
    print("Edge/corner filtering disabled.")

all_rejected = existing_rejected + newly_rejected
MANIFEST.write_text(
    "".join(json.dumps(record) + "\n" for record in kept_records),
    encoding="utf-8",
)
REJECTED_MANIFEST.write_text(
    "".join(json.dumps(record) + "\n" for record in all_rejected),
    encoding="utf-8",
)


ACCEPTED_MANIFEST = MANIFEST
records = kept_records
print("Accepted manifest:", ACCEPTED_MANIFEST)
print("Accepted patches:", len(records))
print("Newly quarantined:", len(newly_rejected))
print("Total quarantined:", len(all_rejected))
if not records:
    raise RuntimeError("The quarantine filter rejected every patch.")


## 5. Deterministic randomized 80/10/10 split within each tile

`spatial_blocks` randomly assigns coarse blocks, reducing leakage versus random individual overlapping patches. This remains an intra-tile comparison, not unseen-geography evaluation.


In [ ]:
from collections import Counter, defaultdict
import hashlib, json, random

SPLIT_SEED = 20260619
SPLIT_MODE = "spatial_blocks"  # or "patch_random" for an explicit leakage-prone ablation
SPATIAL_BLOCK_PIXELS = 2048
FINAL_MANIFEST = WORK_ROOT / "manifest_dgx_80_10_10.jsonl"

def seed_for(tile):
    return int.from_bytes(hashlib.sha256(f"{SPLIT_SEED}:{tile}".encode()).digest()[:8],"little")

def assign(group):
    tile=group[0]["tile_id"]
    if SPLIT_MODE == "patch_random":
        random.Random(seed_for(tile)).shuffle(group)
        n=len(group); ntest=max(1,round(.1*n)); nval=max(1,round(.1*n))
        for i,r in enumerate(group): r["split"]="test" if i<ntest else "val" if i<ntest+nval else "train"
        return group
    blocks=defaultdict(list)
    for r in group: blocks[(r["row"]//SPATIAL_BLOCK_PIXELS,r["col"]//SPATIAL_BLOCK_PIXELS)].append(r)
    keys=list(blocks); random.Random(seed_for(tile)).shuffle(keys); targets={"test":.1*len(group),"val":.1*len(group)}; chosen={}
    for split in ("test","val"):
        count=0
        while keys and (count<targets[split] or count==0):
            key=keys.pop(); chosen[key]=split; count+=len(blocks[key])
    for key in keys: chosen[key]="train"
    output=[]
    for key,values in blocks.items():
        for r in values: r["split"]=chosen[key]; r["split_block"]=[int(key[0]),int(key[1])]; output.append(r)
    return output

accepted=[json.loads(line) for line in ACCEPTED_MANIFEST.read_text(encoding="utf-8").splitlines() if line.strip()]
by_tile=defaultdict(list)
for r in accepted: by_tile[r["tile_id"]].append(dict(r))
records=[]
for tile,group in sorted(by_tile.items()): records.extend(assign(group))
FINAL_MANIFEST.write_text("".join(json.dumps(r)+"\n" for r in records),encoding="utf-8")
MANIFEST = FINAL_MANIFEST
counts=Counter(r["split"] for r in records); per_tile=Counter((r["tile_id"],r["split"]) for r in records)
print("Manifest:",FINAL_MANIFEST,"overall:",counts)
for tile in sorted(by_tile):
    values={s:per_tile[(tile,s)] for s in ("train","val","test")}; total=sum(values.values())
    print(tile,values,{k:round(v/total,3) for k,v in values.items()})
if any(counts[s]==0 for s in ("train","val","test")): raise RuntimeError(f"Missing split: {counts}")


## 6. Visualize accepted HR/LR samples and split geography


In [ ]:
from pathlib import Path
import json
import random
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.nn import functional as F

from geodiff_gan.models.degradation import random_degradation

WORK_ROOT = globals().get("WORK_ROOT", THESIS_ROOT / "geodiff-output")
MANIFEST = globals().get("MANIFEST", FINAL_MANIFEST)
DEGRADATION_SEVERITY = globals().get("DEGRADATION_SEVERITY", "mild")
DEGRADATION_SEED = globals().get("DEGRADATION_SEED", 42)
RANDOM_VISUALIZATION_SEED = globals().get("RANDOM_VISUALIZATION_SEED", 42)
VISUALIZE_ACCEPTED = globals().get("VISUALIZE_ACCEPTED", 5)

accepted_records = [
    json.loads(line)
    for line in MANIFEST.read_text(encoding="utf-8").splitlines()
    if line.strip()
]
randomizer = random.Random(RANDOM_VISUALIZATION_SEED)
selected = randomizer.sample(
    accepted_records, k=min(VISUALIZE_ACCEPTED, len(accepted_records))
)

def chw_to_rgb(tensor):
    return tensor.detach().cpu().clamp(0, 1).permute(1, 2, 0).numpy()

figure, axes = plt.subplots(
    len(selected), 4, figsize=(16, 4 * len(selected)), squeeze=False
)
for row, record in enumerate(selected):
    with np.load(record["patch"]) as data:
        hr = torch.from_numpy(data["hr"]).float()
    if hr.ndim == 3 and hr.shape[-1] == 3:
        hr = hr.permute(2, 0, 1)
    generator = torch.Generator().manual_seed(DEGRADATION_SEED + row)
    observed_lr, parameters, clean_lr = random_degradation(
        hr.unsqueeze(0),
        scale=4,
        generator=generator,
        return_clean=True,
        severity=DEGRADATION_SEVERITY,
    )
    bicubic = F.interpolate(
        observed_lr,
        size=hr.shape[-2:],
        mode="bicubic",
        align_corners=False,
    ).clamp(0, 1)
    noise_l1 = (observed_lr - clean_lr).abs().mean()
    noise_to_signal = noise_l1 / clean_lr.abs().mean().clamp_min(1e-8)
    images = [hr, clean_lr[0], observed_lr[0], bicubic[0]]
    titles = [
        f"HR 10 m\n{record['tile_id']} r{record['row']} c{record['col']}",
        "Clean LR 40 m",
        (
            f"Observed LR ({DEGRADATION_SEVERITY})\n"
            f"noise/signal={float(noise_to_signal):.4f}"
        ),
        f"Bicubic 4x\nparams={parameters[0].tolist()}",
    ]
    for column, (image, title) in enumerate(zip(images, titles)):
        axes[row, column].imshow(chw_to_rgb(image))
        axes[row, column].set_title(title, fontsize=10)
        axes[row, column].axis("off")
figure.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
colors={"train":"#3973ac","val":"#e69f00","test":"#c62828"}
tiles=sorted({r["tile_id"] for r in records})
fig,axes=plt.subplots(len(tiles),1,figsize=(10,4*len(tiles)),squeeze=False)
for ax,tile in zip(axes[:,0],tiles):
    tile_records=[r for r in records if r["tile_id"]==tile]
    for split in ("train","val","test"):
        subset=[r for r in tile_records if r["split"]==split]
        ax.scatter([r["col"] for r in subset],[r["row"] for r in subset],s=16,label=split,color=colors[split])
    ax.invert_yaxis(); ax.set_aspect("equal"); ax.set_title(f"Tile {tile}: spatial split"); ax.legend()
plt.tight_layout(); plt.show()


## 7. Visualize quarantined samples


In [ ]:
from pathlib import Path
import json
import random
import matplotlib.pyplot as plt
import numpy as np

WORK_ROOT = globals().get("WORK_ROOT", THESIS_ROOT / "geodiff-output")
REJECTED_MANIFEST = globals().get(
    "REJECTED_MANIFEST", WORK_ROOT / "rejected-edge-patches.jsonl"
)
BLACK_THRESHOLD = globals().get("BLACK_THRESHOLD", 1e-6)
RANDOM_VISUALIZATION_SEED = globals().get("RANDOM_VISUALIZATION_SEED", 42)
VISUALIZE_REJECTED = globals().get("VISUALIZE_REJECTED", 5)

rejected_records = []
if REJECTED_MANIFEST.exists():
    rejected_records = [
        json.loads(line)
        for line in REJECTED_MANIFEST.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
viewable = [record for record in rejected_records if Path(record["patch"]).exists()]

if not viewable:
    print("No quarantined patch files are available to visualize.")
else:
    randomizer = random.Random(RANDOM_VISUALIZATION_SEED)
    selected_rejected = randomizer.sample(
        viewable, k=min(VISUALIZE_REJECTED, len(viewable))
    )
    figure, axes = plt.subplots(
        len(selected_rejected),
        4,
        figsize=(16, 4 * len(selected_rejected)),
        squeeze=False,
    )
    for row, record in enumerate(selected_rejected):
        with np.load(record["patch"]) as data:
            hr = np.asarray(data["hr"], dtype=np.float32)
            valid_mask = (
                np.asarray(data["valid_mask"]).astype(bool)
                if "valid_mask" in data
                else np.ones(hr.shape[-2:], dtype=bool)
            )
        rgb = np.clip(np.moveaxis(hr, 0, -1), 0, 1)
        black = np.all(hr <= BLACK_THRESHOLD, axis=0)
        overlay = rgb.copy()
        overlay[black] = np.array([1.0, 0.0, 0.0], dtype=np.float32)
        filter_stats = record.get("filter", {})
        displays = [
            (rgb, "Rejected HR"),
            (black, "Black-background mask"),
            (overlay, "Black pixels marked red"),
            (~valid_mask, "Original invalid/SCL mask"),
        ]
        for column, (image, title) in enumerate(displays):
            axes[row, column].imshow(
                image,
                cmap="gray" if image.ndim == 2 else None,
                vmin=0 if image.ndim == 2 else None,
                vmax=1 if image.ndim == 2 else None,
            )
            suffix = ""
            if column == 0:
                suffix = (
                    f"\n{filter_stats.get('reason', 'unknown')}"
                    f" | edge={filter_stats.get('edge_black_fraction', float('nan')):.4f}"
                    f" | corner={filter_stats.get('corner_black_fraction', float('nan')):.4f}"
                )
            axes[row, column].set_title(title + suffix, fontsize=10)
            axes[row, column].axis("off")
    figure.tight_layout()
    plt.show()
    print(f"Showing {len(selected_rejected)} of {len(rejected_records)} rejected records.")


## 8. Caption-free experiment

No captions are generated or loaded. Text conditioning is disabled for every variant.


In [ ]:
CAPTIONS = None
RUN_CAPTIONING = False
print("Caption-free training: no caption file or external text model will be used.")


## 9. Build fair caption-free configs for small-improved, medium, and large

Every config is asserted to use `resize_conv`; none uses PixelShuffle.


In [ ]:
from pathlib import Path
import copy, torch, yaml

VARIANTS_TO_RUN = ["small_improved", "medium", "large"]
STAGES_TO_RUN = ["base", "vae", "diffusion", "joint"]
AUTO_RESUME_TRAINING = True
VALIDATION_LIMIT = 128
EPOCHS_BY_VARIANT = {
    "small_improved":{"base":8,"vae":8,"diffusion":20,"joint":8},
    "medium":{"base":8,"vae":8,"diffusion":20,"joint":8},
    "large":{"base":6,"vae":6,"diffusion":15,"joint":6},
}
BATCH_BY_VARIANT = {
    "small_improved":{"batch":4,"accumulation":4},
    "medium":{"batch":2,"accumulation":8},
    "large":{"batch":1,"accumulation":16},
}
DISCRIMINATOR_CHANNELS = {"small_improved":24, "medium":32, "large":64}

with (REPOSITORY_DIR/"configs/default.yaml").open() as h: defaults=yaml.safe_load(h)
with (REPOSITORY_DIR/"configs/small_12tile_improved.yaml").open() as h: improved=yaml.safe_load(h)
with (REPOSITORY_DIR/"configs/medium.yaml").open() as h: medium=yaml.safe_load(h)
common_training=copy.deepcopy(improved.get("training",{})); common_losses=common_training.pop("loss_weights",{})

def build_config(variant):
    config=copy.deepcopy(defaults)
    if variant=="small_improved": config["model"].update(improved.get("model",{}))
    elif variant=="medium": config["model"].update(medium.get("model",{}))
    elif variant!="large": raise ValueError(variant)
    config["model"]["decoder_upsample_mode"]="resize_conv"
    config["model"]["use_text_conditioning"]=False
    # Hash fallback avoids SigLIP download with older clones; model text input is disabled.
    config["text_encoder"]={"kind":"hash","context_dim":config["model"]["context_dim"],"max_tokens":1}
    config["prompts"]={"null_probability":1.0,"paraphrase_probability":0.0,"mismatch_probability":0.0}
    config["data"].update({"manifest":str(FINAL_MANIFEST),"captions":None,"caption_sampling":"fixed","train_degradation_sampling":"random","degradation_seed":42,"degradation_severity":"mild"})
    config["training"].update(common_training); config["training"]["loss_weights"].update(common_losses)
    config["training"]["discriminator_channels"] = DISCRIMINATOR_CHANNELS[variant]
    config["training"].update({
        "batch_size":BATCH_BY_VARIANT[variant]["batch"],
        "gradient_accumulation":BATCH_BY_VARIANT[variant]["accumulation"],
        "num_workers":8,"amp":True,"gradient_checkpointing":True,"auto_resume":True,
        "progress_mode":"compact","progress_updates_per_epoch":4,"validate_every":1,
        "validation_limit":VALIDATION_LIMIT,"keep_best_and_latest":True,
        "checkpoint_metric":"val_l1","checkpoint_mode":"min",
        "early_stopping_patience":4,"early_stopping_min_epochs":4,"early_stopping_min_delta":0.00005,
    })
    config.setdefault("debug",{}); config["debug"].update({"enabled":False,"print_tensor_stats":False})
    assert config["model"]["decoder_upsample_mode"]=="resize_conv"
    assert config["model"]["use_text_conditioning"] is False
    return config

CONFIGS={name:build_config(name) for name in VARIANTS_TO_RUN}
CONFIG_ROOT=WORK_ROOT/"configs"; RUN_ROOT=WORK_ROOT/"runs"
for variant,config in CONFIGS.items():
    directory=CONFIG_ROOT/variant; directory.mkdir(parents=True,exist_ok=True)
    (directory/"template.yaml").write_text(yaml.safe_dump(config,sort_keys=False),encoding="utf-8")
    print(variant,"batch",config["training"]["batch_size"],"accum",config["training"]["gradient_accumulation"],"upsampler",config["model"]["decoder_upsample_mode"])

from geodiff_gan.parameters import build_parameter_report
train_count=sum(r["split"]=="train" for r in records)
PARAMETER_REPORTS={}
for variant,config in CONFIGS.items():
    report=build_parameter_report(config,patches=train_count,world_size=1); PARAMETER_REPORTS[variant]=report
    print(variant,"core",f"{report['core_model']['scalar_parameters']:,}","joint",f"{report['training_stages']['joint']['total_optimized_parameters']:,}","effective batch",report["training_configuration"]["effective_batch_size"])


## 10. Train variants sequentially with automatic resume

The models are never trained simultaneously. Each stage keeps best/latest checkpoints and resumes from its own directory after interruption.


In [ ]:
import copy, yaml
from pathlib import Path

def select_checkpoint(directory,stage):
    best=directory/f"{stage}_best.pt"
    if best.exists(): return best
    candidates=sorted(directory.glob(f"{stage}_epoch_*.pt"))
    if not candidates: raise RuntimeError(f"No checkpoint for {stage} in {directory}")
    return candidates[-1]

def train_variant(variant):
    previous=None; checkpoints={}
    for stage in STAGES_TO_RUN:
        config=copy.deepcopy(CONFIGS[variant]); output=RUN_ROOT/variant/stage
        config["training"].update({"stage":stage,"epochs":EPOCHS_BY_VARIANT[variant][stage],"output_dir":str(output),"init_checkpoint":str(previous) if previous else None,"resume":None,"auto_resume":AUTO_RESUME_TRAINING})
        if stage=="joint":
            config["training"]["learning_rate"]=1e-5
            config["training"]["loss_weights"]["adversarial"]=0.003
        config_path=CONFIG_ROOT/variant/f"{stage}.yaml"
        config_path.write_text(yaml.safe_dump(config,sort_keys=False),encoding="utf-8")
        print(f"\n===== {variant}: {stage} =====")
        run([PYTHON,"-m","geodiff_gan.cli.train","--config",config_path],cwd=REPOSITORY_DIR)
        previous=select_checkpoint(output,stage); checkpoints[stage]=previous
        print("Selected:",previous)
    return checkpoints

CHECKPOINTS_BY_VARIANT={}
for variant in VARIANTS_TO_RUN:
    CHECKPOINTS_BY_VARIANT[variant]=train_variant(variant)

from IPython.display import Image as DisplayImage, display
for variant in VARIANTS_TO_RUN:
    for stage in STAGES_TO_RUN:
        curve=RUN_ROOT/variant/stage/"training_curves.png"
        if curve.exists():
            print(variant,stage); display(DisplayImage(filename=str(curve)))
CHECKPOINTS_BY_VARIANT


## 11. Evaluate all models on identical test data


In [ ]:
import json, pandas as pd

EVALUATION_LIMIT=40
EVALUATION_SAMPLES=2
EVALUATION_STEPS=20
EVAL_ROOT=WORK_ROOT/"evaluation"
EVALUATION_RESULTS={}
for variant in VARIANTS_TO_RUN:
    config_path=CONFIG_ROOT/variant/"joint.yaml"; checkpoint=CHECKPOINTS_BY_VARIANT[variant]["joint"]; output=EVAL_ROOT/variant/"test"
    run([PYTHON,"-m","geodiff_gan.cli.evaluate","--config",config_path,"--checkpoint",checkpoint,"--output",output,"--split","test","--samples",EVALUATION_SAMPLES,"--steps",EVALUATION_STEPS,"--back-projection-steps",3,"--mode","sr","--limit",EVALUATION_LIMIT,"--device","cuda","--progress","compact","--no-text"],cwd=REPOSITORY_DIR)
    baseline=EVAL_ROOT/variant/"test_baselines.json"
    run([PYTHON,"-m","geodiff_gan.cli.baselines","--config",config_path,"--base-checkpoint",CHECKPOINTS_BY_VARIANT[variant]["base"],"--output",baseline,"--split","test","--limit",EVALUATION_LIMIT,"--device","cuda","--progress","compact"],cwd=REPOSITORY_DIR)
    EVALUATION_RESULTS[variant]={"model":json.loads((output/"metrics.json").read_text()),"baselines":json.loads(baseline.read_text())}
rows=[]
for variant,payload in EVALUATION_RESULTS.items():
    rows.append({"variant":variant,"method":"GeoDiff-GAN",**payload["model"]})
    for method,values in payload["baselines"].items():
        if isinstance(values,dict): rows.append({"variant":variant,"method":method,**values})
metric_table=pd.DataFrame(rows)
columns=[c for c in ["variant","method","count","l1","psnr","ssim","edge_f1","redegradation_l1"] if c in metric_table]
display(metric_table[columns].round(6))
EVAL_ROOT.mkdir(parents=True,exist_ok=True); metric_table.to_csv(EVAL_ROOT/"three_model_comparison.csv",index=False)


In [ ]:
import matplotlib.pyplot as plt
model_metrics=metric_table[metric_table["method"]=="GeoDiff-GAN"].set_index("variant")
fig,axes=plt.subplots(1,4,figsize=(18,4))
for ax,metric,higher in zip(axes,["psnr","ssim","edge_f1","redegradation_l1"],[True,True,True,False]):
    model_metrics[metric].plot(kind="bar",ax=ax,color=["#3973ac","#6a4c93","#c62828"]); ax.set_title(metric+(" ?" if higher else " ?")); ax.tick_params(axis="x",rotation=20)
plt.tight_layout(); plt.show()


## 12. Side-by-side outputs on identical test patches


In [ ]:
import numpy as np, torch
import matplotlib.pyplot as plt
from pathlib import Path
from torch.nn import functional as F
from geodiff_gan.data import SentinelPatchDataset

dataset=SentinelPatchDataset(FINAL_MANIFEST,split="test",scale=4,caption_file=None,augment=False,random_degradation=False,degradation_seed=42,degradation_severity="mild")
for index in range(min(4,len(dataset))):
    sample=dataset[index]; stem=Path(sample["patch"]).stem; hr=sample["hr"]; lr=sample["lr"]
    bicubic=F.interpolate(lr[None],size=hr.shape[-2:],mode="bicubic",align_corners=False)[0].clamp(0,1)
    outputs={}
    for variant in VARIANTS_TO_RUN:
        result=EVAL_ROOT/variant/"test"/f"{stem}_uncertainty.npz"
        if result.exists():
            with np.load(result) as data: outputs[variant]=torch.from_numpy(data["mean"]).float()
    panels=[(F.interpolate(lr[None],size=hr.shape[-2:],mode="nearest")[0],"Input LR"),(bicubic,"Bicubic")]+[(outputs[v],v) for v in VARIANTS_TO_RUN if v in outputs]+[(hr,"Target HR")]
    fig,axes=plt.subplots(2,len(panels),figsize=(4*len(panels),8),squeeze=False)
    for col,(image,title) in enumerate(panels):
        axes[0,col].imshow(image.clamp(0,1).permute(1,2,0)); axes[0,col].set_title(title); axes[0,col].axis("off")
        error=(image-hr).abs().mean(0); axes[1,col].imshow(error,cmap="turbo",vmin=0,vmax=max(.05,float(error.quantile(.99)))); axes[1,col].set_title(f"error mean={float(error.mean()):.4f}"); axes[1,col].axis("off")
    fig.suptitle(stem); plt.tight_layout(); plt.show()


## 13. Detailed diagnostics on the same test index


In [ ]:
from IPython.display import Image as DisplayImage, display
DEBUG_ROOT=WORK_ROOT/"debug_compare"
for variant in VARIANTS_TO_RUN:
    output=DEBUG_ROOT/variant
    run([PYTHON,"-m","geodiff_gan.cli.debug","--config",CONFIG_ROOT/variant/"joint.yaml","--checkpoint",CHECKPOINTS_BY_VARIANT[variant]["joint"],"--output",output,"--split","test","--index",0,"--mode","sr","--steps",EVALUATION_STEPS,"--back-projection-steps",3,"--diffusion-every",5],cwd=REPOSITORY_DIR)
    print("\n=====",variant,"=====")
    for name in ("overview.png","stage_intermediates.png","frequency_spectra.png","edges_and_wavelets.png","policy_overlays.png"):
        path=output/name
        if path.exists(): display(DisplayImage(filename=str(path)))


## 14. Back up results, then clean up dataset

The results archive excludes the dataset. Download it before cleanup.


In [ ]:
from pathlib import Path
import shutil, time
BACKUP_BASE=THESIS_ROOT/f"geodiff-dgx-results-{time.strftime('%Y%m%d-%H%M%S')}"
archive=shutil.make_archive(str(BACKUP_BASE),"zip",WORK_ROOT)
print("Backup:",archive)
run(["du","-sh",WORK_ROOT])
print("Download and verify this archive before deleting DATASET_ROOT.")


In [ ]:
CONFIRM_DATASET_BACKED_UP = False
if CONFIRM_DATASET_BACKED_UP:
    shutil.rmtree(DATASET_ROOT)
    print("Removed:",DATASET_ROOT)
else:
    print("Dataset retained. Set CONFIRM_DATASET_BACKED_UP=True only after backup verification.")
